# Logistic Regression with Hyperparameter Tuning for Emotion Classification
This notebook trains and evaluates a Logistic Regression model for emotion classification on the Dutch GoEmotions dataset using TF-IDF features. It also performs hyperparameter tuning via GridSearchCV to select the best regularization parameters.

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV


In [10]:
# Load training and test data
train_df = pd.read_csv("../data/dataset/processed/go_emotion_dutch.csv")
test_df = pd.read_csv("../data/group_4_url_1_transcript.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Train shape: (57100, 4)
Test shape: (1050, 7)


,Sentence,Emotion_core,value,Emotion_encoded
0,Bedankt! Duidelijke rekwisieten voor hem omdat...,happiness,1,0
1,Goed werk!,happiness,1,0
2,ngl zijn totale diefstal van de lwymmd-teksten...,happiness,1,0
3,Wat een frisse en unieke kritiek!,happiness,1,0
4,alle lof zij de donkere moeder!,happiness,1,0


In [11]:
# Prepare text and labels
train_texts = train_df['Sentence'].astype(str).tolist()
test_texts = test_df['Sentence'].astype(str).tolist()

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(train_df['Emotion_core'].tolist() + test_df['Emotion_core'].tolist())

train_labels = label_encoder.transform(train_df['Emotion_core'].tolist())
test_labels = label_encoder.transform(test_df['Emotion_core'].tolist())

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)
print("Number of classes:", num_classes)


Classes: ['anger' 'disgust' 'fear' 'happiness' 'neutral' 'sadness' 'surprise']
Number of classes: 7


In [12]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

print("Train TF-IDF shape:", X_train.shape)
print("Test TF-IDF shape:", X_test.shape)


Train TF-IDF shape: (57100, 10000)
Test TF-IDF shape: (1050, 10000)


In [13]:
# Logistic Regression with GridSearchCV
clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                         solver='saga', multi_class='multinomial',
                         random_state=42)

# Parameter grid
params = {
    "C": [0.1, 1, 10],
    "penalty": ["l1", "l2"]
}

grid = GridSearchCV(clf, params, cv=5, scoring="f1_weighted", n_jobs=-1)
grid.fit(X_train, train_labels)

print("Best params:", grid.best_params_)
print("Best CV F1 (weighted):", grid.best_score_)


KeyboardInterrupt: 

In [14]:
# Logistic Regression with best parameters
clf = LogisticRegression(
    C=1,
    penalty="l2",
    solver="saga", 
    multi_class="multinomial",
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train
clf.fit(X_train, train_labels)

# Predict
predictions = clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(test_labels, predictions)
f1 = f1_score(test_labels, predictions, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}\n")

print("Classification Report:")
print(classification_report(test_labels, predictions, target_names=label_encoder.classes_))

Test Accuracy: 0.4267
Test F1 Score: 0.4409

Classification Report:
              precision    recall  f1-score   support

       anger       0.15      0.08      0.11        37
     disgust       0.00      0.00      0.00        17
        fear       0.50      0.02      0.03        56
   happiness       0.45      0.41      0.43       198
     neutral       0.62      0.55      0.58       546
     sadness       0.41      0.31      0.35       104
    surprise       0.14      0.35      0.20        92

    accuracy                           0.43      1050
   macro avg       0.32      0.24      0.24      1050
weighted avg       0.49      0.43      0.44      1050



c:\Users\19102\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [15]:
# Use best model to predict on test set
best_model = grid.best_estimator_
predictions = best_model.predict(X_test)

accuracy = accuracy_score(test_labels, predictions)
f1 = f1_score(test_labels, predictions, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}\n")

print("Classification Report:")
print(classification_report(test_labels, predictions, target_names=label_encoder.classes_))


AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'